# 🧪 Compatibility Test Suite — Colab GPU Runner\n\nThis notebook runs the full compatibility test suite on a Colab GPU.\n\n**Requirements:** T4 (16 GB) or better. A100 recommended for 4D models.\n\n---

In [1]:
# ── Probe: Are we on Colab? ──────────────────────────────────────────────────
import os, socket
hostname = socket.gethostname()
is_colab = os.path.exists("/content") or "colab" in hostname.lower()
print(f"Hostname: {hostname}")
print(f"Running on Colab: {is_colab}")
if is_colab:
    print("✅ Connected to Colab runtime!")
    import subprocess
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
    print(f"GPU: {gpu.stdout.strip()}")
else:
    print("❌ NOT on Colab — this is a local kernel.")
    print("   → Click 'Select Kernel' (top-right) → Colab → Auto Connect")

Hostname: cb9f286bb575
Running on Colab: True
✅ Connected to Colab runtime!
GPU: NVIDIA L4, 23034 MiB


In [2]:
# ── Cell 1: Verify GPU & Clone Repo ──────────────────────────────────────────
import subprocess, os

# Check GPU
gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print(f"🖥️  GPU: {gpu_info.stdout.strip()}")

# Clone repo
if not os.path.exists("/content/Research"):
    subprocess.run(["git", "clone", "https://github.com/Snehpatel101/Research.git", "/content/Research"], check=True)
    print("✅ Repo cloned")
else:
    subprocess.run(["git", "-C", "/content/Research", "pull", "--ff-only"], check=True)
    print("✅ Repo updated")

os.chdir("/content/Research")
print(f"📁 Working dir: {os.getcwd()}")

🖥️  GPU: NVIDIA L4, 23034 MiB
✅ Repo cloned
📁 Working dir: /content/Research
✅ Repo cloned
📁 Working dir: /content/Research


In [4]:
# ── Cell 2: Install Dependencies ─────────────────────────────────────────────
import subprocess

subprocess.run(["pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)

# Verify key imports
import torch
print(f"✅ PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Device: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    total = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
    print(f"   VRAM:   {total / 1e9:.1f} GB")

✅ PyTorch 2.8.0+cu126, CUDA available: True
   Device: NVIDIA L4
   VRAM:   23.7 GB


In [5]:
# ── Cell 3: Run Unit Tests (quick sanity check) ─────────────────────────────
import subprocess, os
os.chdir("/content/Research")
os.environ["TORCHDYNAMO_DISABLE"] = "1"

result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-x", "-q", "--tb=short"],
    capture_output=True, text=True, timeout=300
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("⚠️ STDERR:", result.stderr[-2000:])
print(f"\n{'✅' if result.returncode == 0 else '❌'} Unit tests exit code: {result.returncode}")

tests/test_lookahead_audit.py::TestCleanFeature::test_sma_no_lookahead
tests/test_lookahead_audit.py::TestForwardLooking::test_forward_shift_detected
tests/test_lookahead_audit.py::TestForwardLooking::test_raises_in_blocking_mode
tests/test_lookahead_audit.py::TestCorruptionMethods::test_method_detects_lookahead[random]
  /content/Research/src/validation/lookahead_audit.py:376: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[1961.95484858 2240.67322061 5556.25591897 2128.94846331 4057.48470211
   2034.21017076 4165.63465537 3530.53872022 2901.29061738 3035.92107708
   2322.51244052 3601.55277531 2554.65610465 1659.30422005 1598.5022082
   3283.80350418 4917.9574108  3269.16841653 2945.44063579 3415.36997656
   4600.66680435 3338.17205743 2605.84310881 4368.62908223 3581.43435752
   4867.60808728 3296.17322343 1660.62493864 1493.67769995 5001.42126134
   5085.93207482 2874.70583064 2638.07288072 4781.26861

In [7]:
# Quick check: how many passed?
import subprocess, os
os.chdir("/content/Research")
r = subprocess.run(["python", "-m", "pytest", "tests/", "-q", "--tb=no"], capture_output=True, text=True, timeout=300)
lines = r.stdout.strip().split('\n')
for line in lines:
    if 'passed' in line or 'failed' in line or 'error' in line:
        print(line)

  /content/Research/src/validation/lookahead_audit.py:376: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[1961.95484858 2240.67322061 5556.25591897 2128.94846331 4057.48470211


In [8]:
# ── Cell 4: Run Compatibility Tests (--skip-4d, 14 tests) ───────────────────
import subprocess, os, time
os.chdir("/content/Research")
os.environ["TORCHDYNAMO_DISABLE"] = "1"

print("🚀 Starting compatibility test suite (--skip-4d)...")
print("   This tests all 2D + 3D model combinations (14 tests)")
print("   Expected time: ~15-25 min on T4\n")

start = time.time()
result = subprocess.run(
    ["python", "scripts/compatibility_test.py", "--skip-4d"],
    capture_output=True, text=True, timeout=3600
)
elapsed = time.time() - start

print(result.stdout[-5000:] if len(result.stdout) > 5000 else result.stdout)
if result.returncode != 0:
    print("⚠️ STDERR:", result.stderr[-3000:])
print(f"\n⏱️  Total time: {elapsed/60:.1f} min")
print(f"{'✅' if result.returncode == 0 else '❌'} Compatibility tests exit code: {result.returncode}")

🚀 Starting compatibility test suite (--skip-4d)...
   This tests all 2D + 3D model combinations (14 tests)
   Expected time: ~15-25 min on T4

y_entropy: 0.1542
  diversity_kl_divergence: 0.0694

Bundle: /tmp/compat_3D+3D_tcn+lstm_6ub4tyqe/20260302_160208/bundles
Output: /tmp/compat_3D+3D_tcn+lstm_6ub4tyqe/20260302_160208
Starting ML Factory Experiment: compat_test_3D+3D_gru+inceptiontime

[Phase 1/4] Data Pipeline
Running data pipeline...
  Loaded: 6825 rows from /content/Research/data/raw/MES_1m_1week.parquet
  Generating features...
  Features: 227 columns
  Generating labels...
    label_h5: {0: 2156, 1: 80, -1: 56, -99: 5}
  Pipeline complete: 2297 rows, 229 columns
  Label distribution: {0: 2156, 1: 80, -1: 56, -99: 5}
  Data sufficiency check: 2297 bars >= 345 required (embargo=5, purge=10, splits=3) — OK

[Phase 2/4] Model Training
Training 2 models...
  Models: ['gru', 'inceptiontime']
  Mode: standard
  Trained: 2 models
  Best: gru_h5

[Phase 3/4] Evaluation
  Backtest disab

In [10]:
# ── Extract Results ──────────────────────────────────────────────────────────
import re
output = result.stdout + "\n" + result.stderr
# Strip ANSI codes
clean = re.sub(r'\x1b\[[0-9;]*m', '', output)
for line in clean.split('\n'):
    if any(k in line for k in ['PASS', 'FAIL', 'SUMMARY', 'Total:', '/14]']):
        print(line.strip())
print(f"\n⏱️  Total time: {elapsed/60:.1f} min")
print(f"Exit code: {result.returncode}")

[1/14] 2D+2D boosting pair (xgboost + lightgbm, meta=ridge_meta)
15:50:51 [INFO   ] compatibility_test:   ✅ PASS — 2 models, 40.9s
[2/14] 2D+2D catboost+rf (catboost + random_forest, meta=xgboost_meta)
15:51:27 [INFO   ] compatibility_test:   ✅ PASS — 2 models, 34.8s
[3/14] 2D+2D classical pair (logistic + svm, meta=ridge_meta)
15:51:58 [INFO   ] compatibility_test:   ✅ PASS — 2 models, 30.8s
[4/14] 2D+3D xgb+tcn (xgboost + tcn, meta=ridge_meta)
15:53:02 [INFO   ] compatibility_test:   ✅ PASS — 2 models, 62.9s
[5/14] 2D+3D lgb+lstm (lightgbm + lstm, meta=xgboost_meta)
15:54:03 [INFO   ] compatibility_test:   ✅ PASS — 2 models, 60.4s
[6/14] 2D+3D xgb+gru (xgboost + gru, meta=ridge_meta)
15:55:07 [INFO   ] compatibility_test:   ✅ PASS — 2 models, 62.9s
[7/14] 2D+3D lgb+transformer (lightgbm + transformer, meta=ridge_meta)
15:56:12 [INFO   ] compatibility_test:   ✅ PASS — 2 models, 63.1s
[8/14] 2D+3D xgb+tft (xgboost + tft, meta=ridge_meta)
15:57:43 [ERROR  ] compatibility_test:   ❌ FAIL 

In [16]:
# ── Run TFT solo with full traceback ─────────────────────────────────────────
import subprocess, os
os.chdir("/content/Research")

code = '''
import sys, os, traceback, tempfile, logging
os.environ["TORCHDYNAMO_DISABLE"] = "1"
logging.basicConfig(level=logging.WARNING)

sys.path.insert(0, ".")
from pathlib import Path
from src.config.data import FeatureConfig, LabelingConfig, SequenceConfig
from src.config.experiment import (
    BundlingSection, DataSection, EvaluationSection,
    ExperimentConfig, TrainingSection,
)
from src.config.training import CalibrationConfig, OptunaConfig
from src.factory import MLFactory

with tempfile.TemporaryDirectory(prefix="tft_debug_") as tmpdir:
    config = ExperimentConfig(
        name="tft_debug",
        output_dir=Path(tmpdir),
        data=DataSection(
            symbol="MES",
            data_path="data/raw/MES_1m_1week.parquet",
            features=FeatureConfig(families=["price", "volume", "volatility"]),
            labeling=LabelingConfig(method="triple_barrier"),
            sequence=SequenceConfig(seq_len=30),
        ),
        training=TrainingSection(
            models=["xgboost", "tft"],
            horizons=[5],
            training_mode="standard",
            n_splits=3,
            purge_bars=10,
            embargo_bars=5,
            build_ensemble=True,
            meta_learner="ridge_meta",
            max_epochs=3,
            batch_size=64,
            optuna=OptunaConfig(n_trials=0),
            calibration=CalibrationConfig(enabled=False),
        ),
        evaluation=EvaluationSection(run_backtest=False, compute_shap=False),
        bundling=BundlingSection(create_bundle=True, deploy_artifact=False),
    )
    try:
        factory = MLFactory(config)
        result = factory.run()
        print(f"SUCCESS: {result}")
    except Exception as e:
        traceback.print_exc()
'''
r = subprocess.run(
    ["python", "-c", code],
    capture_output=True, text=True, timeout=300,
    env={**os.environ, "TORCHDYNAMO_DISABLE": "1"}
)
combined = (r.stdout + "\n" + r.stderr).strip().split('\n')
# Print last 60 lines
for line in combined[-60:]:
    print(line)

Model Performance:
  xgboost_h5: F1=0.0000, Acc=0.0000
  tft_h5: F1=0.0000, Acc=0.0000

Bundle: /tmp/tft_debug_36obb5wo/20260302_161558/bundles
Output: /tmp/tft_debug_36obb5wo/20260302_161558
SUCCESS: ExperimentResult(run_id='20260302_161558', config=ExperimentConfig(name='tft_debug', description='', run_id='20260302_161558', output_dir=PosixPath('/tmp/tft_debug_36obb5wo/20260302_161558'), random_seed=42, verbose=1, data=DataSection(symbol='MES', data_path=PosixPath('data/raw/MES_1m_1week.parquet'), start_date=None, end_date=None, features=FeatureConfig(mode='full', families=['price', 'volume', 'volatility'], sma_periods=[10, 20, 50, 100, 200], ema_periods=[9, 21, 50], atr_periods=[7, 14, 21], rsi_period=14, macd_params={'fast': 12, 'slow': 26, 'signal': 9}, bb_period=20, bb_std=2.0, selection_enabled=True, selection_method='mda', selection_n_features=50, selection_cv_splits=5, selection_min_frequency=0.6), labeling=LabelingConfig(method='triple_barrier', upper_mult=2.0, lower_mult=2.0

In [17]:
# ── Filter TFT error output ──────────────────────────────────────────────────
combined = (r.stdout + "\n" + r.stderr).strip().split('\n')
# Show errors, tracebacks, and key lines
for line in combined:
    low = line.lower()
    if any(k in low for k in ['error', 'traceback', 'exception', 'fail', 'oom', 'cuda', 'memory', 'success', 'file "', 'raise ']):
        print(line.rstrip()[:300])
print(f"\nExit code: {r.returncode}")
print(f"Stdout lines: {len(r.stdout.split(chr(10)))}")
print(f"Stderr lines: {len(r.stderr.split(chr(10)))}")

ML Factory Experiment: SUCCESS
SUCCESS: ExperimentResult(run_id='20260302_161558', config=ExperimentConfig(name='tft_debug', description='', run_id='20260302_161558', output_dir=PosixPath('/tmp/tft_debug_36obb5wo/20260302_161558'), random_seed=42, verbose=1, data=DataSection(symbol='MES', data_path=PosixPath('data/raw/MES_1m_1wee
ERROR:src.models.neural.oom_recovery:OOM recovery failed: max retries (3) exceeded
ERROR:src.models.training.services.ensemble_service:Need at least 2 models for ensemble, got 1: ['xgboost_h5']

Exit code: 0
Stdout lines: 78
Stderr lines: 42


## ⬇️ Optional: Full Suite with 4D Models (A100 recommended)\nOnly run the cell below if you have an A100 (40 GB+). PatchTST and iTransformer need significant VRAM.

In [ ]:
# ── Cell 5: Run FULL Compatibility Tests (all 17 tests, A100 only) ──────────
import subprocess, os, time
os.chdir("/content/Research")
os.environ["TORCHDYNAMO_DISABLE"] = "1"

print("🚀 Starting FULL compatibility test suite (all 17 tests incl. 4D)...")
print("   ⚠️  Requires A100 (40 GB+) for PatchTST / iTransformer")
print("   Expected time: ~30-45 min on A100\n")

start = time.time()
result = subprocess.run(
    ["python", "scripts/compatibility_test.py"],
    capture_output=True, text=True, timeout=7200
)
elapsed = time.time() - start

print(result.stdout[-5000:] if len(result.stdout) > 5000 else result.stdout)
if result.returncode != 0:
    print("⚠️ STDERR:", result.stderr[-3000:])
print(f"\n⏱️  Total time: {elapsed/60:.1f} min")
print(f"{'✅' if result.returncode == 0 else '❌'} FULL test suite exit code: {result.returncode}")